# Feature Engineering for Appliance Recognition

Companion notebook for Chapter 2. Builds the weighted recurrence graph
(WRG), adaptive WRG, and Fortescue-transform features on real PLAID and
LILACD data, then trains a real CNN classifier to compare them. See the
chapter text for the full narrative and citations; this notebook is the
code and results behind it.

Data comes from `resources/nilm-code/AWRGNILM` (the author's own research
code, gitignored in this repo). Re-running this notebook end to end
requires that directory to be present locally.

In [ ]:
from lets_plot import (
    LetsPlot,
    aes,
    coord_fixed,
    facet_wrap,
    geom_bar,
    geom_line,
    geom_path,
    geom_text,
    geom_tile,
    gggrid,
    ggplot,
    ggsize,
    labs,
    scale_color_manual,
    scale_fill_gradientn,
    scale_fill_manual,
)
import numpy as np
import pandas as pd
from scipy.signal import medfilt
from sklearn.metrics import confusion_matrix, f1_score, matthews_corrcoef
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn

from ark.plot.theme import modern_theme
from ark.plot.tokens import BLUES_SEQUENTIAL, BRAND_PALETTE, GRAY_400

LetsPlot.setup_html()

In [ ]:
def paa(series: np.ndarray, w: int) -> np.ndarray:
    """Reduce a 1D signal to `w` points via Piecewise Aggregate Approximation.

    Splits `series` into `w` equal segments and replaces each with its mean,
    the dimensionality-reduction step between a raw activation window and
    the distance matrix built from it.

    Args:
        series: 1D array of length T_s (the raw activation window).
        w: Target length after reduction (the embedding size).

    Returns:
        1D array of length `w`.
    """
    n = len(series)
    bounds = np.linspace(0, n, w + 1).astype(int)
    return np.array([series[bounds[k] : bounds[k + 1]].mean() for k in range(w)])


def distance_matrix(x: np.ndarray) -> np.ndarray:
    """Build the pairwise absolute-distance matrix D of a 1D signal.

    Args:
        x: 1D array of length w (already PAA-reduced).

    Returns:
        w x w array where entry [i, j] is |x[i] - x[j]|.
    """
    return np.abs(x[:, None] - x[None, :])


def weighted_recurrence(dist: np.ndarray, lam: float, delta: float) -> np.ndarray:
    """Turn a distance matrix into a weighted recurrence graph (WRG).

    Implements r_ij = min(floor(dist_ij * lam), delta): scales the distance,
    floors it to an integer, and caps it at `delta`. Setting delta=1
    recovers the binary recurrence graph.

    Args:
        dist: w x w distance matrix.
        lam: Scale parameter, lam = 1 / epsilon.
        delta: Maximum value a cell can take.

    Returns:
        w x w array with values in [0, delta].
    """
    return np.minimum(np.floor(dist * lam), delta)


def vi_image(current: np.ndarray, voltage: np.ndarray, w: int) -> np.ndarray:
    """Rasterize a current/voltage trajectory into a binary w x w V-I image.

    Centers and bins the current and voltage onto a w x w grid, marking a
    cell wherever the trajectory passes through it. This is the baseline
    feature representation WRG is compared against.

    Args:
        current: 1D array of length w (PAA-reduced activation current).
        voltage: 1D array of length w (PAA-reduced activation voltage).
        w: Grid resolution (must match the length of `current`/`voltage`).

    Returns:
        w x w array, 1 where the trajectory passes through a cell.
    """
    c = np.clip(current, -np.abs(current).max(), np.abs(current).max())
    v = np.clip(voltage, -np.abs(voltage).max(), np.abs(voltage).max())
    dc = (c.max() - c.min()) / (w - 1)
    dv = (v.max() - v.min()) / (w - 1)
    ci = np.clip(((c - c.min()) / dc).astype(int), 0, w - 1)
    vi = np.clip(((v - v.min()) / dv).astype(int), 0, w - 1)
    img = np.zeros((w, w))
    for a, b in zip(ci, vi, strict=False):
        img[a, w - b - 1] += 1
    return img


def matrix_to_long(mat: np.ndarray, name: str) -> pd.DataFrame:
    """Reshape a w x w image matrix into a long-format DataFrame for plotting.

    Args:
        mat: w x w array (a distance matrix, RG, WRG, or V-I image).
        name: Label identifying this matrix, used to facet or compare
            multiple representations in the same plot.

    Returns:
        DataFrame with columns x, y, value, representation, one row per
        matrix cell, ready for `geom_tile`.
    """
    n = mat.shape[0]
    rows, cols = np.meshgrid(np.arange(n), np.arange(n), indexing="ij")
    return pd.DataFrame(
        {
            "x": cols.ravel(),
            "y": (n - 1 - rows).ravel(),
            "value": mat.ravel(),
            "representation": name,
        }
    )

## Load one real activation event and build its WRG

Loads a single, real PLAID sub-metered activation current/voltage window
and builds the distance matrix, binary recurrence graph (RG), and weighted
recurrence graph (WRG) from it, alongside the V-I trajectory/image
baseline.

In [ ]:
plaid_dir = "../../resources/nilm-code/AWRGNILM/data/plaid/submetered/"
current = np.load(plaid_dir + "current.npy")
voltage = np.load(plaid_dir + "voltage.npy")
labels = np.load(plaid_dir + "labels.npy")

example = np.where(labels == 3)[0][0]
i_raw = medfilt(current[example], kernel_size=5)
v_raw = medfilt(voltage[example], kernel_size=5)

waveform_df = pd.DataFrame({"sample": np.arange(len(i_raw)), "current": i_raw})

waveform_plot = (
    ggplot(waveform_df, aes("sample", "current"))
    + geom_line(color=BRAND_PALETTE[0], size=0.9)
    + labs(
        title="Activation current",
        subtitle="A real PLAID sub-metered event",
        x="Sample",
        y="Current (A)",
    )
    + modern_theme()
    + ggsize(330, 330)
)

In [ ]:
trajectory_df = pd.DataFrame({"voltage": v_raw, "current": i_raw})

trajectory_plot = (
    ggplot(trajectory_df, aes("voltage", "current"))
    + geom_path(color=BRAND_PALETTE[4], size=0.9)
    + labs(
        title="V-I trajectory",
        subtitle="The same event, current against voltage",
        x="Voltage (V)",
        y="Current (A)",
    )
    + modern_theme()
    + ggsize(330, 330)
)

gggrid([waveform_plot, trajectory_plot], ncol=2)

In [ ]:
w = 50
i_paa = paa(i_raw, w)
v_paa = paa(v_raw, w)
D = distance_matrix(i_paa)

lam, delta = 10.0, 20.0
rg = (1.0 / lam <= D).astype(float)
wrg = weighted_recurrence(D, lam, delta)

rg_wrg_df = pd.concat(
    [
        matrix_to_long(rg, "Binary recurrence graph"),
        matrix_to_long(wrg, "Weighted recurrence graph"),
    ]
)

rg_wrg_plot = (
    ggplot(rg_wrg_df, aes("x", "y", fill="value"))
    + geom_tile()
    + facet_wrap("representation")
    + scale_fill_gradientn(colors=BLUES_SEQUENTIAL)
    + coord_fixed(ratio=1)
    + labs(title="Thresholding throws away exactly the detail that separates appliances")
    + modern_theme(show_x_axis=False, legend_pos="none")
    + ggsize(560, 320)
)
rg_wrg_plot

In [ ]:
vi = vi_image(i_paa, v_paa, w)

baseline_df = pd.concat(
    [
        matrix_to_long(vi, "V-I image"),
        matrix_to_long(wrg, "Weighted recurrence graph"),
    ]
)

baseline_plot = (
    ggplot(baseline_df, aes("x", "y", fill="value"))
    + geom_tile()
    + facet_wrap("representation")
    + scale_fill_gradientn(colors=BLUES_SEQUENTIAL)
    + coord_fixed(ratio=1)
    + labs(
        title="WRG keeps structure the V-I image throws away",
        subtitle="Two feature representations, same activation event",
    )
    + modern_theme(show_x_axis=False, legend_pos="none")
    + ggsize(560, 320)
)
baseline_plot

## Phase imbalance: one real LILACD three-phase event

Loads a real LILACD bulb activation (three-phase aggregate current) and
applies the Fortescue transform, isolating the zero-sequence component that
carries the imbalance caused by a single-phase appliance sharing a line
with larger three-phase loads.

In [ ]:
lilac_dir = "../../resources/nilm-code/AWRGNILM/data/lilac/aggregated/"
lilac_current = np.load(lilac_dir + "current.npy")
lilac_labels = np.load(lilac_dir + "labels.npy")

bulb_example = np.where(lilac_labels == "Bulb")[0][0]
i3 = np.array([medfilt(phase, kernel_size=5) for phase in lilac_current[bulb_example]])

phase_df = pd.concat(
    [pd.DataFrame({"sample": np.arange(i3.shape[1]), "current": i3[k], "phase": f"Phase {k + 1}"}) for k in range(3)]
)

phase_plot = (
    ggplot(phase_df, aes("sample", "current", color="phase"))
    + geom_line(size=0.8)
    + scale_color_manual(values=BRAND_PALETTE)
    + labs(
        title="A single-phase bulb, wired to one leg of a three-phase system",
        subtitle="Real LILACD aggregate current: one phase carries the bulb's load, the other two carry almost nothing",
        x="Sample",
        y="Current (A)",
    )
    + modern_theme(legend_pos="bottom")
    + ggsize(650, 320)
)
phase_plot

In [ ]:
alpha = np.exp(1j * 2 * np.pi / 3)
fortescue = np.array([[1, alpha, alpha**2], [1, alpha**2, alpha], [1, 1, 1]]) * (2 / 3)

sequence = (fortescue @ i3).real
sequence_names = ["Positive sequence", "Negative sequence", "Zero sequence"]

sequence_df = pd.concat(
    [
        pd.DataFrame(
            {
                "sample": np.arange(i3.shape[1]),
                "current": sequence[k],
                "component": sequence_names[k],
            }
        )
        for k in range(3)
    ]
)

sequence_plot = (
    ggplot(sequence_df, aes("sample", "current", color="component"))
    + geom_line(size=0.8)
    + scale_color_manual(values=BRAND_PALETTE)
    + labs(
        title="The zero-sequence component carries the imbalance",
        subtitle="Fortescue transform: zero-sequence appears only because the load is unbalanced",
        x="Sample",
        y="Current (A)",
    )
    + modern_theme(legend_pos="bottom")
    + ggsize(650, 320)
)
sequence_plot

## Train and evaluate a real classifier on the full dataset

Builds V-I, WRG, and WRG+FT features for every sample in the full LILACD
dataset (all 16 appliance types, three phases stacked as channels), then
trains and evaluates the same small CNN on each, so the comparison is real
and reproducible rather than only cited from the papers.

In [ ]:
class ApplianceCNN(nn.Module):
    """Small 3-layer CNN appliance classifier.

    The same architecture used for every feature representation in this
    chapter (V-I image, WRG, WRG+FT): three convolutional layers followed
    by two fully connected layers, matching AWRGNILM's `Conv2D` model.

    Args:
        n_classes: Number of appliance categories to predict.
        width: Height/width of the square input image (w x w).
        in_channels: Number of input channels, one per phase (3 for
            three-phase LILACD, 1 for single-phase data).
    """

    def __init__(self, n_classes: int, width: int, in_channels: int = 3):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 16, 5, 2),
            nn.ReLU(),
            nn.Conv2d(16, 32, 5, 2),
            nn.ReLU(),
            nn.Conv2d(32, 64, 5, 2),
            nn.ReLU(),
        )
        with torch.no_grad():
            flat_size = self.conv(torch.zeros(1, in_channels, width, width)).numel()
        self.classifier = nn.Sequential(
            nn.Linear(flat_size, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, n_classes)
        )

    def forward(self, x):
        """Predict class logits for a batch of images.

        Args:
            x: Tensor of shape (batch, in_channels, width, width).

        Returns:
            Tensor of shape (batch, n_classes) of unnormalized logits.
        """
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


def train_and_evaluate(features: np.ndarray, y: np.ndarray, width: int, epochs: int = 40):
    """Train an `ApplianceCNN` on one feature representation and evaluate it.

    Splits the data 75/25 (stratified, matching the AWRG paper's protocol),
    trains on the 75% split, and scores the held-out 25%.

    Args:
        features: Array of shape (n_samples, channels, width, width), one
            of vi_features, wrg_features, or wrg_ft_features.
        y: Integer class label per sample, shape (n_samples,).
        width: Height/width of each feature image (must match `features`).
        epochs: Number of full-batch gradient steps to train for.

    Returns:
        Dict with macro F1 (`f1`), Matthews correlation coefficient
        (`mcc`), and the held-out `y_test`/`y_pred` arrays (for a
        confusion matrix).
    """
    x_train, x_test, y_train, y_test = train_test_split(features, y, test_size=0.25, stratify=y, random_state=42)
    x_train_t = torch.tensor(x_train).float()
    y_train_t = torch.tensor(y_train).long()
    x_test_t = torch.tensor(x_test).float()

    torch.manual_seed(0)
    model = ApplianceCNN(n_classes=len(np.unique(y)), width=width)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    model.train()
    for _ in range(epochs):
        optimizer.zero_grad()
        loss = loss_fn(model(x_train_t), y_train_t)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        y_pred = model(x_test_t).argmax(dim=1).numpy()

    f1 = f1_score(y_test, y_pred, average="macro") * 100
    mcc = matthews_corrcoef(y_test, y_pred)
    return {"f1": f1, "mcc": mcc, "y_test": y_test, "y_pred": y_pred}

In [ ]:
lilac_voltage = np.load(lilac_dir + "voltage.npy")

classes = sorted(np.unique(lilac_labels))
class_to_idx = {name: idx for idx, name in enumerate(classes)}
y = np.array([class_to_idx[label] for label in lilac_labels])

n_samples = lilac_current.shape[0]
vi_features = np.zeros((n_samples, 3, w, w), dtype=np.float32)
wrg_features = np.zeros((n_samples, 3, w, w), dtype=np.float32)
wrg_ft_features = np.zeros((n_samples, 3, w, w), dtype=np.float32)

for idx in range(n_samples):
    sequence_components = (fortescue @ lilac_current[idx]).real
    for phase in range(3):
        i_phase = paa(lilac_current[idx, phase], w)
        v_phase = paa(lilac_voltage[idx, phase], w)
        vi_features[idx, phase] = vi_image(i_phase, v_phase, w)
        wrg_features[idx, phase] = weighted_recurrence(distance_matrix(i_phase), lam, delta)

        seq_phase = paa(sequence_components[phase], w)
        wrg_ft_features[idx, phase] = weighted_recurrence(distance_matrix(seq_phase), lam, delta)

In [ ]:
vi_result = train_and_evaluate(vi_features, y, width=w)
wrg_result = train_and_evaluate(wrg_features, y, width=w)
wrg_ft_result = train_and_evaluate(wrg_ft_features, y, width=w)

comparison_df = pd.DataFrame(
    {
        "representation": ["V-I image", "WRG", "WRG + FT"],
        "f1": [vi_result["f1"], wrg_result["f1"], wrg_ft_result["f1"]],
    }
)

comparison_plot = (
    ggplot(comparison_df, aes("representation", "f1", fill="representation"))
    + geom_bar(stat="identity", width=0.6)
    + geom_text(aes(label="f1"), format="{.1f}", va="bottom", nudge_y=1.5, size=8)
    + scale_fill_manual(values=[GRAY_400, GRAY_400, BRAND_PALETTE[4]])
    + labs(
        title="WRG, then the Fortescue transform, each raise accuracy",
        subtitle="Macro F1 on a held-out 25% of LILACD, same CNN, three different inputs",
        x="",
        y="Macro F1 (%)",
    )
    + modern_theme(legend_pos="none")
    + ggsize(500, 340)
)
comparison_plot

In [ ]:
cm = confusion_matrix(wrg_ft_result["y_test"], wrg_ft_result["y_pred"])
cm_df = pd.DataFrame(cm, index=classes, columns=classes)
cm_long = cm_df.reset_index().melt(id_vars="index", var_name="predicted", value_name="count")
cm_long = cm_long.rename(columns={"index": "actual"})
cm_labels = cm_long[cm_long["count"] > 0]

confusion_plot = (
    ggplot(cm_long, aes("predicted", "actual", fill="count"))
    + geom_tile()
    + geom_text(aes(label="count"), data=cm_labels, color="#1E293B", size=7)
    + scale_fill_gradientn(colors=BLUES_SEQUENTIAL)
    + coord_fixed(ratio=1)
    + labs(
        title="Most remaining confusions are between similar resistive appliances",
        subtitle="WRG + FT confusion matrix, held-out test set",
        x="Predicted",
        y="Actual",
    )
    + modern_theme(x_axis_angle=90, legend_pos="none")
    + ggsize(560, 560)
)
confusion_plot